In [1]:
# Imports to do the math
import pandas as pd
import numpy as np
from sklearn.decomposition import PCA

In [2]:
# Get the data and organize
df = pd.read_csv('first_25000_rows.csv')
df.sort_values(['ts_event','symbol'], inplace=True)
df.reset_index(drop=True, inplace=True)

In [ ]:
# Helper function to calculate OFI

def compute_ofi(group):
    prev = {m: {'bid_px':np.nan,'bid_sz':np.nan,'ask_px':np.nan,'ask_sz':np.nan} for m in range(10)}
    rec = []
    for _, row in group.iterrows():
        ofi_b = np.zeros(10)
        ofi_a = np.zeros(10)
        for m in range(10):
            pb0,pb1 = prev[m]['bid_px'],     row[f'bid_px_{m:02d}']
            qb0,qb1 = prev[m]['bid_sz'],     row[f'bid_sz_{m:02d}']
            pa0,pa1 = prev[m]['ask_px'],     row[f'ask_px_{m:02d}']
            qa0,qa1 = prev[m]['ask_sz'],     row[f'ask_sz_{m:02d}']
            if np.isnan(pb0) or pb1>pb0:      ofi_b[m] =  qb1
            elif pb1< pb0:                    ofi_b[m] = -qb0
            else:                             ofi_b[m] =  qb1 - qb0
            if np.isnan(pa0) or pa1>pa0:      ofi_a[m] = -qa1
            elif pa1< pa0:                    ofi_a[m] =  qa0
            else:                             ofi_a[m] =  qa1 - qa0
            prev[m] = {'bid_px':pb1,'bid_sz':qb1,'ask_px':pa1,'ask_sz':qa1}
        rec.append((ofi_b,ofi_a))
    return rec

In [4]:
all_ofs = []
for sym, grp in df.groupby('symbol', sort=False):
    ofs = compute_ofi(grp)
    all_ofs.extend(ofs)

ofi_b = np.vstack([o[0] for o in all_ofs])
ofi_a = np.vstack([o[1] for o in all_ofs])

In [5]:
# Best level OFI
df['best_level_ofi'] = ofi_b[:,0] - ofi_a[:,0]

In [6]:
# Multi level OFI
ofis = ofi_b - ofi_a
df[[f'ofi_L{m}' for m in range(10)]] = ofis

depth_cols = [f'bid_sz_{m:02d}' for m in range(10)] + [f'ask_sz_{m:02d}' for m in range(10)]
df['avg_depth'] = df[depth_cols].sum(axis=1)/10

for m in range(10):
    df[f'ofi_norm_L{m}'] = df[f'ofi_L{m}'] / df['avg_depth']

In [7]:
# Integrated OFI
X = df[[f'ofi_norm_L{m}' for m in range(10)]].fillna(0)
pca = PCA(n_components=1).fit(X)
w1 = pca.components_[0]
l1 = np.abs(w1).sum()
df['integrated_ofi'] = X.values.dot(w1) / l1

In [9]:
# Cross Asset OFI
df['cross_asset_ofi'] = df.groupby('ts_event')['best_level_ofi'].transform(lambda x: x.sum() - x)

ofis = df[['ts_event','symbol','best_level_ofi']
          + [f'ofi_L{m}' for m in range(10)]
          + [f'ofi_norm_L{m}' for m in range(10)]
          + ['integrated_ofi','cross_asset_ofi']]

ofis.to_csv('ofi_features.csv', index=False)